# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  
**`Student Name`:** Ananya Singla   
**`Roll Number`:** U20230077   
**`GitHub Branch`:** ananya_U20230077  

# Imports and Setup

In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler


# Load Datasets

In [44]:
# Load datasets
news_df = pd.read_csv("data/news_articles.csv")
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

print(news_df.head())
print(train_users.head())


                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to predict...  Carla K. Johnson, AP   
1  He was subdued by passengers and crew when he ...        Mary Papenfuss   
2  "Until you have a dog y

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [45]:
import re

news_df = news_df.dropna()

median_ages_by_label = train_users.groupby('label')['age'].median()
for label in train_users['label'].unique():
    mask = (train_users['label'] == label) & (train_users['age'].isnull())
    train_users.loc[mask, 'age'] = median_ages_by_label[label]
test_users['age'].fillna(train_users['age'].median(), inplace=True)

le = LabelEncoder()
train_users["user_class"] = le.fit_transform(train_users["label"])

train_users["subscriber"] = train_users["subscriber"].astype(int)
test_users["subscriber"] = test_users["subscriber"].astype(int)

region_freq = train_users['region_code'].value_counts().to_dict()
train_users['region_freq'] = train_users['region_code'].map(region_freq)
test_users['region_freq'] = test_users['region_code'].map(region_freq).fillna(1)

def extract_version(v):
    m = re.match(r'(\d+)\.(\d+)', str(v))
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)

train_v = train_users['browser_version'].apply(extract_version)
test_v = test_users['browser_version'].apply(extract_version)
train_users['browser_major'] = train_v.apply(lambda x: x[0])
train_users['browser_minor'] = train_v.apply(lambda x: x[1])
test_users['browser_major'] = test_v.apply(lambda x: x[0])
test_users['browser_minor'] = test_v.apply(lambda x: x[1])

/var/folders/fr/x19nqp_n1kj5j4zdgrxschk00000gn/T/ipykernel_94396/1434754731.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_users['age'].fillna(train_users['age'].median(), inplace=True)


In [46]:
X = train_users.drop(columns=["label", "user_class", "region_code", "browser_version"]).select_dtypes(include=[np.number])
y = train_users["user_class"]

X = pd.DataFrame(X)
X['spend_per_transaction'] = X['avg_monthly_spend'] / (X['num_transactions'] + 1)
X['engagement_per_click'] = X['engagement_score'] / (X['clicks'] + 1)
X['value_per_view'] = X['avg_cart_value'] / (X['product_views'] + 1)
X['purchase_rate'] = X['purchase_amount'] / (X['session_duration'] + 1)
X['interaction_rate'] = X['interaction_count'] / (X['time_on_site'] + 1)
X['loyalty_engagement'] = X['loyalty_index'] * X['engagement_score']
X['spend_variety'] = X['avg_monthly_spend'] * X['content_variety']
X['engagement_depth'] = X['engagement_score'] * X['browsing_depth']

imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
key_features = ['engagement_score', 'avg_monthly_spend', 'loyalty_index', 'revisit_rate', 'region_freq']
key_idx = [list(X.columns).index(f) for f in key_features if f in X.columns]
X_poly = poly.fit_transform(X_imputed[:, key_idx])
X_imputed = np.hstack([X_imputed, X_poly[:, len(key_idx):]])

In [47]:
X_train, X_val, y_train, y_val = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.


Develop a classification model (e.g., Decision Tree, Logistic Regression) to predict the user category
(User1, User2, or User3) based on input feature data.
•Split the train_users.csv dataset into a training set (80%) and a validation set (20%) for
model evaluation.
•The model must be trained on the training set and evaluated on the validation set to ensure it
can accurately classify users into their respective categories.
•This classifier will serve as the “Context Detector” for your bandit system where you will use
test_users.csv dataset.

In [48]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import classification_report

rf = RandomForestClassifier(n_estimators=250, max_depth=14, min_samples_split=4, random_state=42)
gb = GradientBoostingClassifier(n_estimators=250, max_depth=5, learning_rate=0.05, subsample=0.8, random_state=42)

clf = VotingClassifier(estimators=[('rf', rf), ('gb', gb)], voting='soft')
clf.fit(X_train, y_train)

y_pred = clf.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred, target_names=le.classes_))

Accuracy: 0.955
              precision    recall  f1-score   support

      user_1       0.91      0.97      0.94       142
      user_2       0.99      0.90      0.94       142
      user_3       0.97      1.00      0.99       116

    accuracy                           0.95       400
   macro avg       0.96      0.96      0.96       400
weighted avg       0.96      0.95      0.95       400



# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
